In [22]:
#!/usr/bin/env python

from tqdm import tqdm
from pathlib import Path
import py7zr
import tempfile
import gzip
import shutil

import convert_bxsf
from mc3d_web_pipeline.s3_utils import upload_file_to_s3

BXSF_LOCATION_ROOT = Path("/home/eimre_k/mcxd/fermi-surfaces/test_data_mc3d/stash_bxsf")
BXSF_ARCHIVE_NAME = "bxsf.7z"
BXSF_FILE_NAME = "wjl.bxsf"

ENERGY_WINDOW = [-0.5, 0.5] # eV
RESOLUTION = 20

S3_BUCKET = "mc-discover-mcxd-public"
S3_PREFIX = "mc3d/pbesol-v1_fermisurf"

In [23]:
def get_bxsf_path_from_uuid(uuid):
    """The path is sharded based on the 4 digits of the uuid"""
    part1 = uuid[:2]
    part2 = uuid[2:4]
    part3 = uuid[4:]
    return BXSF_LOCATION_ROOT / part1 / part2 / part3 / BXSF_ARCHIVE_NAME


def extract_and_process(path: Path, mc3d_id: str):
    """Extract the .7z to a temp folder, generate the json, gzip and upload."""
    if not path.exists():
        raise FileNotFoundError(path)

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)

        with py7zr.SevenZipFile(path, mode="r") as z:
            z.extractall(path=tmpdir)

        bxsf_path = tmpdir / BXSF_FILE_NAME

        if not bxsf_path.exists():
            raise FileNotFoundError(f"{BXSF_FILE_NAME} was not found after extraction")

        json_path = tmpdir / f"{mc3d_id}.json"

        convert_bxsf.prepare_json(
            bxsf_path,
            energy_window=ENERGY_WINDOW,
            resolution=RESOLUTION,
            output_path=json_path
        )

        gz_path = json_path.with_suffix(json_path.suffix + ".gz")
        with json_path.open("rb") as f_in, gzip.open(gz_path, "wb", compresslevel=6) as f_out:
            shutil.copyfileobj(f_in, f_out)

        if not gz_path.exists():
            raise RuntimeError(f"Gzip step failed to produce: {gz_path}")

        s3_extras = {
            "ContentType": "application/json",
            "ContentEncoding": "gzip",
        }
        upload_file_to_s3(
            gz_path, S3_BUCKET, f"{S3_PREFIX}/{mc3d_id}.json", s3_extras
        )


In [24]:
from mc3d_web_pipeline.mc_data_injector import McDataInjector
mdi = McDataInjector()
base_coll = mdi.db["pbesol-v1_fermisurf_base"]


MongoDB connection successful.


In [ ]:
# Prepare the json, upload to S3.
# check at https://rgw.cscs.ch/matcloud:mc-discover-mcxd-public/mc3d/pbesol-v1_fermisurf/mc3d-55055.json

ids_and_wjl_uuids = list(base_coll.find({}, {"wannierjl_calcjob_uuid": 1}))

for entry in tqdm(ids_and_wjl_uuids):
    mc3d_id = entry["_id"]
    wjl_uuid = entry["wannierjl_calcjob_uuid"]

    path = get_bxsf_path_from_uuid(wjl_uuid)
    extract_and_process(path, mc3d_id)

  2%|▏         | 10/407 [00:08<04:46,  1.39it/s]

In [19]:
# test upload
path = Path("./test.json.gz")
extra = {
    "ContentType": "application/json",
    "ContentEncoding": "gzip",
}
upload_file_to_s3(
    path, "mc-discover-mcxd-public", "mc3d/test/test.json", extra
)
# https://rgw.cscs.ch/matcloud:mc-discover-mcxd-public/mc3d/test/test.json